In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.cuda.amp import autocast

print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime > Change runtime type > Select GPU")

GPU Available: True
GPU Name: NVIDIA A100-SXM4-80GB
GPU Memory: 79.32 GB


In [ ]:
torch.cuda.get_device_capability(0)[0]

8

In [ ]:
def make_batch(batch_size, n_ctx, d, noise_std=0.05, w_std=1.0):
    B = batch_size
    n = n_ctx
    Din = d + 1
    L = 2 * n + 1

    w = torch.randn(B, d) * w_std

    X = torch.randn(B, n, d)
    y = (X * w[:, None, :]).sum(dim=-1) + noise_std * torch.randn(B, n)

    xq = torch.randn(B, d)
    yq = (xq * w).sum(dim=-1) + noise_std * torch.randn(B)

    seq = torch.zeros(B, L, Din)
    types = torch.zeros(B, L, dtype=torch.long)

    for i in range(n):
        xi_pos = 2 * i
        yi_pos = 2 * i + 1
        seq[:, xi_pos, :d] = X[:, i, :]
        seq[:, yi_pos, -1] = y[:, i]
        types[:, yi_pos] = 1

    seq[:, -1, :d] = xq
    types[:, -1] = 0
    return seq, types, yq

In [ ]:
def make_batch_xy(batch_size: int, n_ctx: int, d: int, noise_std: float = 0.05, w_std: float = 1.0):
    """
    Returns:
      seq: (B, L, Din) where L = n_ctx + 1 and Din = d + 1
           tokens 0..n_ctx-1 are [x_i, y_i]
           token  n_ctx    is [x_query, 0]
      y_q: (B,) target y for the query
    """
    B = batch_size
    L = n_ctx + 1
    Din = d + 1

    # Per-episode random linear model
    w = torch.randn(B, d) * w_std

    # Context data
    X = torch.randn(B, n_ctx, d)
    y = (X * w[:, None, :]).sum(dim=-1) + noise_std * torch.randn(B, n_ctx)

    # Query
    xq = torch.randn(B, d)
    yq = (xq * w).sum(dim=-1) + noise_std * torch.randn(B)

    # Build tokens
    seq = torch.zeros(B, L, Din)
    seq[:, :n_ctx, :d] = X
    seq[:, :n_ctx, -1] = y
    seq[:, -1, :d] = xq
    seq[:, -1, -1] = 0.0

    return seq, yq

In [ ]:
def make_batch_xy_padded(
    batch_size: int,
    d: int,
    n_ctx_max: int = 128,
    noise_std: float = 0.05,
    w_std: float = 1.0,
    n_ctx_min: int = 2,
):
    B = batch_size
    L = n_ctx_max + 1
    Din = d + 1

    # sample per-episode context length
    n_ctx = torch.randint(low=n_ctx_min, high=n_ctx_max + 1, size=(B,))

    # per-episode regression weights
    w = torch.randn(B, d) * w_std

    X = torch.randn(B, n_ctx_max, d)
    y = (X * w[:, None, :]).sum(dim=-1) + noise_std * torch.randn(B, n_ctx_max)

    idx = torch.arange(n_ctx_max).unsqueeze(0)
    real = idx < n_ctx.unsqueeze(1)

    # allocate tensors
    seq = torch.zeros(B, L, Din)
    pad_mask = torch.ones(B, L, dtype=torch.bool)

    seq[:, :n_ctx_max, :d] = X
    seq[:, :n_ctx_max, -1] = y
    pad_mask[:, :n_ctx_max] = ~real

    # fill query token (always present at last position)
    xq = torch.randn(B, d)
    yq = (xq * w).sum(dim=-1) + noise_std * torch.randn(B)

    seq[:, -1, :d] = xq
    pad_mask[:, -1] = False

    seq[:, :n_ctx_max, :] *= real.unsqueeze(-1)

    return seq, yq, pad_mask, n_ctx

In [ ]:
class ICLTransformer(nn.Module):
    def __init__(self, d_in: int, d_model: int, n_heads: int, n_layers: int, max_len: int):
        super().__init__()
        self.in_proj = nn.Linear(d_in, d_model)

        self.type_emb = nn.Embedding(2, d_model)       # type 0=x, 1=y
        self.pos_emb = nn.Embedding(max_len, d_model)  # positional

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, seq, pad_mask=None):
        """
        seq:   (B,L,d_in)
        returns: (B,) predicted y at the last position
        """
        B, L, _ = seq.shape
        pos = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)

        types = torch.zeros(B, L, dtype=torch.long, device=seq.device)
        types[:, -1] = 1

        h = self.in_proj(seq) + self.pos_emb(pos) + self.type_emb(types)

        h = self.encoder(h, mask=None, src_key_padding_mask=pad_mask)

        yhat = self.out(h[:, -1, :]).squeeze(-1)
        return yhat

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

d = 5
n_ctx_max = 128
L = n_ctx_max + 1
d_in = d + 1

model = ICLTransformer(d_in=d_in, d_model=256, n_heads=8, n_layers=6, max_len=L).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:
steps = 15000
batch_size = 256

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

for step in range(1, steps + 1):
    seq, yq, pad_mask, n_ctx = make_batch_xy_padded(
        batch_size=batch_size,
        d=d,
        n_ctx_max=n_ctx_max,
        n_ctx_min=2,
        noise_std=0.05,
    )
    seq = seq.to(device)
    yq = yq.to(device)
    pad_mask = pad_mask.to(device)

    # pred = model(seq, pad_mask=pad_mask)
    # loss = F.mse_loss(pred, yq)

    # opt.zero_grad(set_to_none=True)

    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
        pred = model(seq, pad_mask=pad_mask)
        loss = F.mse_loss(pred, yq)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    if step % 500 == 0:
        print(f"step {step:5d} | train RMSE: {loss.sqrt().item():.4f} | mean n_ctx: {n_ctx.float().mean().item():.1f}")

step   500 | train RMSE: 0.4227 | mean n_ctx: 65.4
step  1000 | train RMSE: 0.4490 | mean n_ctx: 63.5
step  1500 | train RMSE: 0.4117 | mean n_ctx: 66.5
step  2000 | train RMSE: 0.4069 | mean n_ctx: 65.9
step  2500 | train RMSE: 0.3329 | mean n_ctx: 65.6
step  3000 | train RMSE: 0.3686 | mean n_ctx: 68.3
step  3500 | train RMSE: 0.2755 | mean n_ctx: 64.3
step  4000 | train RMSE: 0.2690 | mean n_ctx: 63.9
step  4500 | train RMSE: 0.2436 | mean n_ctx: 66.4
step  5000 | train RMSE: 0.2296 | mean n_ctx: 68.0
step  5500 | train RMSE: 0.3276 | mean n_ctx: 66.1
step  6000 | train RMSE: 0.3212 | mean n_ctx: 65.9
step  6500 | train RMSE: 0.4181 | mean n_ctx: 63.2
step  7000 | train RMSE: 0.4182 | mean n_ctx: 64.4
step  7500 | train RMSE: 0.2198 | mean n_ctx: 65.7
step  8000 | train RMSE: 0.3341 | mean n_ctx: 62.3
step  8500 | train RMSE: 0.4196 | mean n_ctx: 63.9
step  9000 | train RMSE: 0.3455 | mean n_ctx: 65.0
step  9500 | train RMSE: 0.4546 | mean n_ctx: 66.7
step 10000 | train RMSE: 0.4720

## Evals

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def ols_predict_from_seq(seq: torch.Tensor, d: int, n_ctx: int, ridge: float = 0.0):
    """
    seq: (B, n_ctx+1, d+1) with context tokens [x_i, y_i] and query token [x_q, 0]
    Returns: (B,) OLS (or ridge) predictions for y_q
    """
    B, L, Din = seq.shape
    assert L == n_ctx + 1 and Din == d + 1

    X = seq[:, :n_ctx, :d]          # (B,n,d)
    y = seq[:, :n_ctx, -1]          # (B,n)
    xq = seq[:, -1, :d]             # (B,d)

    # Solve (X^T X + λI) w = X^T y per episode
    yhat = torch.empty(B, device=seq.device, dtype=seq.dtype)
    I = torch.eye(d, device=seq.device, dtype=seq.dtype)

    for b in range(B):
        XtX = X[b].T @ X[b]                     # (d,d)
        Xty = X[b].T @ y[b]                     # (d,)
        if ridge > 0:
            w_hat = torch.linalg.solve(XtX + ridge * I, Xty)
        else:
            # Use lstsq for stability
            w_hat = torch.linalg.lstsq(X[b], y[b]).solution
        yhat[b] = (xq[b] * w_hat).sum()

    return yhat


@torch.no_grad()
def eval_icl_suite(model, make_batch_xy, device, d, n_ctx, batch_size=512, noise_std=0.05, ridge=0.0):
    """
    Runs:
      - normal evaluation
      - shuffle context token order
      - permute y within context (breaks x-y pairing)
      - wipe context (should collapse)
      - OLS/Ridge comparison
    Prints RMSEs and a couple of diagnostics.
    """
    model.eval()

    # ----- base batch -----
    seq, yq = make_batch_xy(batch_size, n_ctx, d, noise_std=noise_std)
    seq, yq = seq.to(device), yq.to(device)

    # types currently unused by your model, but forward expects it
    types = torch.zeros(seq.shape[:2], dtype=torch.long, device=device)

    pred = model(seq, types)
    rmse = F.mse_loss(pred, yq).sqrt().item()

    # ----- shuffle context order (permute tokens 0..n_ctx-1) -----
    perm = torch.randperm(n_ctx, device=device)
    seq_shuf = seq.clone()
    seq_shuf[:, :n_ctx, :] = seq_shuf[:, perm, :]
    pred_shuf = model(seq_shuf, types)
    rmse_shuf = F.mse_loss(pred_shuf, yq).sqrt().item()

    # ----- break x-y pairing: permute y among context tokens, keep X fixed -----
    seq_bad = seq.clone()
    perm_y = torch.randperm(n_ctx, device=device)
    seq_bad[:, :n_ctx, -1] = seq_bad[:, perm_y, -1]  # shuffle y column only
    pred_bad = model(seq_bad, types)
    rmse_bad = F.mse_loss(pred_bad, yq).sqrt().item()

    # ----- wipe context entirely: keep only query x -----
    seq_wipe = seq.clone()
    seq_wipe[:, :n_ctx, :] = 0.0
    pred_wipe = model(seq_wipe, types)
    rmse_wipe = F.mse_loss(pred_wipe, yq).sqrt().item()

    # ----- OLS / ridge baseline -----
    yhat_ols = ols_predict_from_seq(seq, d=d, n_ctx=n_ctx, ridge=ridge)
    rmse_ols = F.mse_loss(yhat_ols, yq).sqrt().item()

    # ----- context sensitivity (mean abs pred change) -----
    delta_wipe = (pred - pred_wipe).abs().mean().item()
    delta_bad = (pred - pred_bad).abs().mean().item()

    print(f"RMSE (model, normal):          {rmse:.4f}")
    print(f"RMSE (model, shuffled ctx):    {rmse_shuf:.4f}")
    print(f"RMSE (model, y-permuted ctx):  {rmse_bad:.4f}")
    print(f"RMSE (model, wiped ctx):       {rmse_wipe:.4f}")
    print(f"RMSE (OLS{' ridge' if ridge>0 else ''}):               {rmse_ols:.4f}")
    print()
    print(f"Mean |pred - pred_wipe|:       {delta_wipe:.4f}")
    print(f"Mean |pred - pred_yperm|:      {delta_bad:.4f}")

    return {
        "rmse": rmse,
        "rmse_shuf": rmse_shuf,
        "rmse_bad": rmse_bad,
        "rmse_wipe": rmse_wipe,
        "rmse_ols": rmse_ols,
        "delta_wipe": delta_wipe,
        "delta_bad": delta_bad,
    }


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
out = eval_icl_suite(model, make_batch_xy, device=device, d=d, n_ctx=n_ctx, batch_size=1024, noise_std=0.05)
print(out)


NameError: name 'make_batch_xy' is not defined

In [ ]:
@torch.no_grad()
def ols_predict_from_seq(seq: torch.Tensor, d: int, n_ctx: int):
    B, L, Din = seq.shape
    assert L == n_ctx + 1 and Din == d + 1
    X = seq[:, :n_ctx, :d]
    y = seq[:, :n_ctx, -1]
    xq = seq[:, -1, :d]

    yhat = torch.empty(B, device=seq.device, dtype=seq.dtype)
    for b in range(B):
        w_hat = torch.linalg.lstsq(X[b], y[b]).solution
        yhat[b] = (xq[b] * w_hat).sum()
    return yhat


@torch.no_grad()
def eval_rmse(model, d, n_ctx, batch_size=2048, noise_std=0.05):
    seq, yq = make_batch_xy(batch_size, n_ctx, d, noise_std=noise_std)
    seq, yq = seq.to(device), yq.to(device)
    pred = model(seq)
    rmse_model = F.mse_loss(pred, yq).sqrt().item()

    pred_ols = ols_predict_from_seq(seq, d=d, n_ctx=n_ctx)
    rmse_ols = F.mse_loss(pred_ols, yq).sqrt().item()
    return rmse_model, rmse_ols

for n in [2, 4, 8, 12, 16, 24, 32, 48, 64]:
    rm_model, rm_ols = eval_rmse(model, d=d, n_ctx=n, batch_size=2048, noise_std=0.05)
    print(f"n_ctx={n:2d} | model RMSE={rm_model:.4f} | OLS RMSE={rm_ols:.4f}")


NameError: name 'make_batch_xy' is not defined

In [ ]:
@torch.no_grad()
def make_batch_xy_fixed_padded(batch_size: int, n_ctx: int, d: int, n_ctx_max: int = 128, noise_std: float = 0.05, w_std: float = 1.0):
    assert n_ctx <= n_ctx_max
    B = batch_size
    L = n_ctx_max + 1
    Din = d + 1

    w = torch.randn(B, d) * w_std

    X = torch.randn(B, n_ctx, d)
    y = (X * w[:, None, :]).sum(dim=-1) + noise_std * torch.randn(B, n_ctx)

    xq = torch.randn(B, d)
    yq = (xq * w).sum(dim=-1) + noise_std * torch.randn(B)

    seq = torch.zeros(B, L, Din)
    pad_mask = torch.ones(B, L, dtype=torch.bool)

    seq[:, :n_ctx, :d] = X
    seq[:, :n_ctx, -1] = y
    pad_mask[:, :n_ctx] = False

    seq[:, -1, :d] = xq
    pad_mask[:, -1] = False

    return seq, yq, pad_mask


@torch.no_grad()
def ols_predict_from_fixed(seq: torch.Tensor, d: int, n_ctx: int):
    X = seq[:, :n_ctx, :d]
    y = seq[:, :n_ctx, -1]
    xq = seq[:, -1, :d]
    B = seq.shape[0]
    yhat = torch.empty(B, device=seq.device, dtype=seq.dtype)
    for b in range(B):
        w_hat = torch.linalg.lstsq(X[b], y[b]).solution
        yhat[b] = (xq[b] * w_hat).sum()
    return yhat

@torch.no_grad()
def sweep(model, d, n_ctx_list, batch_size=4096, n_ctx_max=128):
    model.eval()
    for n in n_ctx_list:
        seq, yq, pad_mask = make_batch_xy_fixed_padded(batch_size, n, d, n_ctx_max=n_ctx_max, noise_std=0.05)
        seq, yq, pad_mask = seq.to(device), yq.to(device), pad_mask.to(device)

        pred = model(seq, pad_mask=pad_mask)
        rm_model = F.mse_loss(pred, yq).sqrt().item()

        pred_ols = ols_predict_from_fixed(seq, d=d, n_ctx=n)
        rm_ols = F.mse_loss(pred_ols, yq).sqrt().item()

        print(f"n_ctx={n:3d} | model RMSE={rm_model:.4f} | OLS RMSE={rm_ols:.4f}")

sweep(model, d, [2,4,8,12,16,24,32,48,64,96,128], batch_size=4096, n_ctx_max=128)


n_ctx=  2 | model RMSE=1.8737 | OLS RMSE=1.7337
n_ctx=  4 | model RMSE=1.2800 | OLS RMSE=1.1471
n_ctx=  8 | model RMSE=0.3980 | OLS RMSE=0.0929
n_ctx= 12 | model RMSE=0.2302 | OLS RMSE=0.0689
n_ctx= 16 | model RMSE=0.1831 | OLS RMSE=0.0610
n_ctx= 24 | model RMSE=0.1506 | OLS RMSE=0.0578
n_ctx= 32 | model RMSE=0.1315 | OLS RMSE=0.0539
n_ctx= 48 | model RMSE=0.1173 | OLS RMSE=0.0539
n_ctx= 64 | model RMSE=0.1065 | OLS RMSE=0.0518
n_ctx= 96 | model RMSE=0.1004 | OLS RMSE=0.0520
n_ctx=128 | model RMSE=0.0983 | OLS RMSE=0.0517
